[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matrixmultiplyconsortium/matrix-compute-colab/blob/main/03_confidential_computing_tee_demo.ipynb)

# 03 — Confidential Computing (TEE): Multi-Vendor Hardware Attestation & AES-256-GCM Encrypted Dispatch

When multiplying proprietary weights or regulated financial/biomedical tensors on decentralized GPUs, **integrity verification (Freivalds) is not enough** — you also need **confidentiality** so neither the relay, network observers, nor the host OS operator can inspect your matrices.

In this notebook, you will explore the `matrix-compute` **Confidential Computing (TEE)** pipeline:
1. **Multi-Vendor Hardware Attestation Appraisal**: Verify X.509 certificate chains, cryptographic quotes, and Golden Reference Integrity Measurements (RIM) across **NVIDIA Hopper CC (`nvidia-cca`)**, **AMD SEV-SNP (`amd-sev-snp`)**, and **Intel TDX (`intel-tdx`)**.
2. **Fail-Closed Adversarial Rejection**: Watch the `AttestationVerifier` reject tampered firmware measurements, replayed nonces, and man-in-the-middle (MITM) public key substitutions before any data leaves the client.
3. **Ephemeral X25519 ECDH + AES-256-GCM Tile Streaming**: Execute an end-to-end encrypted distributed matrix multiplication and verify that wire ciphertexts exhibit near-ideal Shannon entropy ($H \ge 7.95\text{ bits/byte}$).

In [1]:
# Step 1: Install matrix-compute (if running on a fresh Google Colab instance)
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("matrix_compute") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matrix-compute"])

import math
from collections import Counter
import numpy as np

from matrix_compute import (
    AttestationBindingError,
    AttestationMeasurementMismatchError,
    AttestationNonceMismatchError,
    AttestationVerifier,
    MatrixComputeClient,
    VerificationPolicy,
)
from matrix_compute.crypto.pki_fixtures import (
    DEFAULT_GOLDEN_RIM,
    DEFAULT_SNP_GOLDEN_RIM,
    DEFAULT_TDX_GOLDEN_RIM,
    TAMPERED_MEASUREMENTS,
    create_valid_attestation_evidence,
    create_valid_intel_tdx_evidence,
    create_valid_sev_snp_evidence,
    get_default_amd_pki,
    get_default_intel_pki,
    get_default_pki,
)
from matrix_compute.crypto.session import (
    generate_challenger_nonce,
    generate_ephemeral_keypair,
)

print("Confidential Computing (TEE) verification modules loaded!")

Confidential Computing (TEE) verification modules loaded!


## Step 2: Multi-Vendor Hardware Attestation Appraisal (NVIDIA H100 CC, AMD SEV-SNP, Intel TDX)

Before streaming any matrix tiles, the client generates a 32-byte cryptographic `attestation_nonce` and challenges the worker enclave. The enclave signs an attestation report binding:
- Its hardware measurement registers (GPU driver/vBIOS RIMs for **NVIDIA CC**, 48-byte `LAUNCH_DIGEST` for **AMD SEV-SNP**, or `MRTD`/`RTMR[0..3]` for **Intel TDX**),
- The client's `attestation_nonce` (preventing replay attacks), and
- The enclave's ephemeral X25519 public key (preventing MITM TLS/gRPC termination).

In [2]:
# Configure a strict fail-closed VerificationPolicy trusting vendor Root CAs and Golden RIMs
policy = VerificationPolicy()
policy.add_trusted_root_pem(get_default_pki().root_cert_pem)
policy.add_trusted_root_pem(get_default_amd_pki().root_cert_pem)
policy.add_trusted_root_pem(get_default_intel_pki().root_cert_pem)

policy.add_golden_rim("nvidia-cca", DEFAULT_GOLDEN_RIM)
policy.add_golden_rim("amd-sev-snp", DEFAULT_SNP_GOLDEN_RIM)
policy.add_golden_rim("intel-tdx", DEFAULT_TDX_GOLDEN_RIM)

verifier = AttestationVerifier(policy=policy)

nonce = generate_challenger_nonce()
_, worker_eph_pub = generate_ephemeral_keypair()

enclave_evidences = [
    ("NVIDIA H100 Hopper CC", create_valid_attestation_evidence(nonce=nonce, worker_ephemeral_pubkey=worker_eph_pub, provider="nvidia-cca")),
    ("AMD EPYC Milan/Genoa SEV-SNP", create_valid_sev_snp_evidence(nonce=nonce, worker_ephemeral_pubkey=worker_eph_pub)),
    ("Intel Xeon 5th-Gen TDX", create_valid_intel_tdx_evidence(nonce=nonce, worker_ephemeral_pubkey=worker_eph_pub)),
]

print(f"{'Hardware Enclave Architecture':<30} | {'Provider ID':<12} | {'Cert Chain':>10} | {'RIM Digests':>11} | {'Status'}")
print("-" * 86)
for arch_name, evidence in enclave_evidences:
    res = verifier.verify(
        evidence=evidence,
        expected_nonce=nonce,
        expected_client_pubkey=worker_eph_pub,
        task_id="tee-handshake-demo",
    )
    print(
        f"{arch_name:<30} | {res.provider:<12} | {res.chain_length:>5d} certs | "
        f"{res.measurements_verified:>5d} match | VERIFIED (Root: {res.root_cert.subject.rfc4514_string()[:28]}...)"
    )

Hardware Enclave Architecture  | Provider ID  | Cert Chain | RIM Digests | Status
--------------------------------------------------------------------------------------
NVIDIA H100 Hopper CC          | nvidia-cca   |     3 certs |     5 match | VERIFIED (Root: CN=NVIDIA Confidential Compu...)
AMD EPYC Milan/Genoa SEV-SNP   | amd-sev-snp  |     3 certs |     2 match | VERIFIED (Root: CN=ARK-Milan,O=Advanced Micr...)
Intel Xeon 5th-Gen TDX         | intel-tdx    |     3 certs |     5 match | VERIFIED (Root: CN=Intel SGX Root CA,O=Intel...)


## Step 3: Fail-Closed Adversarial Tamper Rejection

What happens if a malicious worker operator modifies the container/driver binary, replays an old attestation quote from another node, or attempts a Man-in-the-Middle (MITM) key substitution? Let's test all three attack vectors against `AttestationVerifier`.

In [3]:
# Attack 1: Tampered GPU Driver / Firmware Measurement (Golden RIM Mismatch)
tampered_rim_evidence = create_valid_attestation_evidence(
    nonce=nonce,
    worker_ephemeral_pubkey=worker_eph_pub,
    provider="nvidia-cca",
    measurements=TAMPERED_MEASUREMENTS,
)
try:
    verifier.verify(tampered_rim_evidence, expected_nonce=nonce, expected_client_pubkey=worker_eph_pub)
except AttestationMeasurementMismatchError as err:
    print(f"[Attack 1 Blocked — Tampered Firmware RIM] -> {type(err).__name__}: {err}")

# Attack 2: Replay Attack (Stale Challenger Nonce)
stale_nonce = generate_challenger_nonce()
try:
    verifier.verify(enclave_evidences[0][1], expected_nonce=stale_nonce, expected_client_pubkey=worker_eph_pub)
except AttestationNonceMismatchError as err:
    print(f"[Attack 2 Blocked — Replayed Quote Nonce]  -> {type(err).__name__}: {err}")

# Attack 3: Host-OS MITM Ephemeral Key Substitution
_, attacker_eph_pub = generate_ephemeral_keypair()
try:
    verifier.verify(enclave_evidences[1][1], expected_nonce=nonce, expected_client_pubkey=attacker_eph_pub)
except AttestationBindingError as err:
    print(f"[Attack 3 Blocked — MITM Key Substitution] -> {type(err).__name__}: {err}")

[Attack 1 Blocked — Tampered Firmware RIM] -> AttestationMeasurementMismatchError: Golden RIM measurement appraisal failed: 1 mismatches, 0 missing (provider='nvidia-cca', mismatches={'driver': {'expected': '7f83b1657ff1fc53b92dc18148a1d65dfc2d4b1fa3d677284addd200126d9069', 'actual': 'deadbeef000000000000000000000000000000000000000000000000deadbeef'}}, missing_claims=[])
[Attack 2 Blocked — Replayed Quote Nonce]  -> AttestationNonceMismatchError: Challenger nonce mismatch: expected 6d7d0f77a4c6a0646b2488f1329777d8bbab7edc3f6a636d2825a0546b8134b6, got a50748c6956cb0e972b835370d85fcac92f19cf8bd19cd197b721c4583e7cbe0 (provider='nvidia-cca', expected_nonce='6d7d0f77a4c6a0646b2488f1329777d8bbab7edc3f6a636d2825a0546b8134b6', actual_nonce='a50748c6956cb0e972b835370d85fcac92f19cf8bd19cd197b721c4583e7cbe0')
[Attack 3 Blocked — MITM Key Substitution] -> AttestationBindingError: Client ephemeral public key binding mismatch: expected 4e71a514db4d27fee0258add2efb76009e785ea7d50d0f3f40234f4bf10de0ad

## Step 4: End-to-End AES-256-GCM Encrypted GEMM & Wire Entropy Analysis

Now let's run an end-to-end confidential matrix multiplication with `require_confidential_compute=True`.
To prove that the matrix tiles are encrypted on the wire, we construct low-entropy structured matrices $A$ and $B$ (containing mostly repeated constants so raw plaintext has very low Shannon entropy $H < 2.0\text{ bits/byte}$) and measure the byte-level Shannon entropy

$$H(X) = -\sum_{b=0}^{255} P(b) \log_2 P(b)$$

of the actual wire payload chunks captured at the gRPC transport layer.

In [4]:
def shannon_entropy_bits_per_byte(data: bytes) -> float:
    if not data:
        return 0.0
    counts = Counter(data)
    total = len(data)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

# Construct a highly structured (low-entropy) 256x256 FP32 matrix
A_secret = np.full((256, 256), 0.5, dtype=np.float32)
A_secret[np.arange(256), np.arange(256)] = 2.0
B_secret = np.triu(np.ones((256, 256), dtype=np.float32))
plaintext_bytes = A_secret.tobytes() + B_secret.tobytes()
plaintext_entropy = shannon_entropy_bits_per_byte(plaintext_bytes)

print(f"Plaintext Tensor Size     : {len(plaintext_bytes) / 1024:.1f} KiB")
print(f"Plaintext Shannon Entropy : {plaintext_entropy:.4f} bits/byte (Highly structured / compressible)\n")

for tee_arch in ("nvidia-cca", "amd-sev-snp", "intel-tdx"):
    with MatrixComputeClient(
        mode="trial",
        require_confidential_compute=True,
        required_tee_type=tee_arch,
    ) as tee_client:
        C_confidential = tee_client.matmul(A_secret, B_secret, tile_size=256)
        wire_chunks = []
        for w in tee_client.mock_workers:
            wire_chunks.extend(w.servicer.last_wire_ciphertext_chunks)
            if w.servicer.last_encrypted_result_bytes:
                wire_chunks.append(w.servicer.last_encrypted_result_bytes)
        wire_blob = b"".join(wire_chunks)
        wire_entropy = shannon_entropy_bits_per_byte(wire_blob)
        max_err = float(np.max(np.abs(C_confidential - (A_secret @ B_secret))))
        print(
            f"[{tee_arch:<11}] Wire Ciphertext: {len(wire_blob) / 1024:>6.1f} KiB | "
            f"Shannon Entropy: {wire_entropy:.4f} / 8.0000 bits/byte | "
            f"Decrypted Max Error: {max_err:.2e}"
        )
        assert wire_entropy >= 7.90, f"Expected high ciphertext entropy, got {wire_entropy}"
        assert np.allclose(C_confidential, A_secret @ B_secret)

print("\nAll confidential TEE dispatches verified: zero plaintext leakage on wire and exact decrypted results!")

Plaintext Tensor Size     : 512.0 KiB
Plaintext Shannon Entropy : 1.0204 bits/byte (Highly structured / compressible)

[nvidia-cca ] Wire Ciphertext:  768.0 KiB | Shannon Entropy: 7.9998 / 8.0000 bits/byte | Decrypted Max Error: 0.00e+00
[amd-sev-snp] Wire Ciphertext:  768.0 KiB | Shannon Entropy: 7.9997 / 8.0000 bits/byte | Decrypted Max Error: 0.00e+00
[intel-tdx  ] Wire Ciphertext:  768.0 KiB | Shannon Entropy: 7.9997 / 8.0000 bits/byte | Decrypted Max Error: 0.00e+00

All confidential TEE dispatches verified: zero plaintext leakage on wire and exact decrypted results!
